# =====================================================
# AI Code Generation Pipeline - Python Track (Project CodeNet)
# =====================================================


In [1]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm pyarrow datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00


In [12]:
import os
import torch
import random
import re
import pandas as pd
from tqdm import tqdm
from google.colab import drive
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Code_Datasets/"
os.makedirs(DRIVE_DIR, exist_ok=True)
OUTPUT_PYTHON = os.path.join(DRIVE_DIR, "synthetic_codenet_python.parquet")

print("Initializing DeepSeek-Coder-7B-Instruct-v1.5 with 4-bit Quantization...")
model_id = "deepseek-ai/deepseek-coder-7b-instruct-v1.5"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initializing DeepSeek-Coder-7B-Instruct-v1.5 with 4-bit Quantization...


config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [7]:
def clean_bpe_artifacts(raw_text):
    if not isinstance(raw_text, str):
        return raw_text
    clean_text = raw_text.replace("Ġ", " ").replace("Ċ", "\n")
    clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
    return clean_text.strip()

def generate_code_variant(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.85,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    generated_text = clean_bpe_artifacts(generated_text)

    extracted_code = generated_text
    if "```" in generated_text:
        try:
            extracted_code = generated_text.split(f"```python")[1].split("```")[0].strip()
        except IndexError:
            try:
                extracted_code = generated_text.split("```")[1].split("```")[0].strip()
            except IndexError:
                pass

    return extracted_code.strip()

In [15]:
from datasets import load_dataset

dataset = load_dataset("claudios/code_search_net", "python", split="train")

all_python_functions = dataset["func_code_string"]

print(type(all_python_functions))
print(len(all_python_functions))

<class 'datasets.arrow_dataset.Column'>
412178


In [ ]:
def synthesize_python_codenet(num_files_needed=25000):
    print("\n--- Running Colab Integrated Python (CodeNet) Track ---")
    print("Downloading/Loading Project CodeNet Python split...")
    dataset = load_dataset("claudios/code_search_net", "python", split="train")
    all_python_functions = dataset['func_code_string']

    if os.path.exists(OUTPUT_PYTHON):
        print("Found existing Python Parquet checkpoint! Resuming...")
        df_existing = pd.read_parquet(OUTPUT_PYTHON)
        synthetic_records = df_existing.to_dict("records")
        start_index = len(synthetic_records)
        print(f"Resuming from index {start_index}...")
    else:
        print("Starting fresh Python task.")
        synthetic_records = []
        start_index = 0

    counter = start_index
    pbar = tqdm(total=num_files_needed, initial=start_index, desc="Synthesizing Python Codes")

    while counter < num_files_needed:
        selected_samples = random.sample(all_python_functions, k=3)

        concatenated_block = ""
        for idx, func_sample in enumerate(selected_samples):
            safe_sample = str(func_sample)[:1000] # OOM Protection
            concatenated_block += f"\n--- Python Sample {idx+1} ---\n{safe_sample}\n"

        prompt = f"""
        You are an Expert Python Developer. Examine this cluster of Python functions written to solve similar tasks.

        Cluster of Reference Python Implementations:
        {concatenated_block}

        Write a completely brand new, highly robust, and distinct Python function that satisfies the same functional objective.

        Strict Operational Rules:
        1. Use Pythonic conventions (list comprehensions, generators, type hints if applicable).
        2. Fully rename all local parameters and logic routines.
        3. Do not mirror any single reference layout. Heavy refactoring is strictly required.
        4. Respond with NOTHING else except the code block inside a single ```python ... ``` block.
        """

        ai_code = generate_code_variant(prompt)

        synthetic_records.append({
            "code": ai_code,
            "language": "python",
            "id": counter
        })
        counter += 1
        pbar.update(1)

        # Batch Saving every 50 files
        if len(synthetic_records) % 50 == 0:
            pd.DataFrame(synthetic_records).to_parquet(OUTPUT_PYTHON, index=False)

    pd.DataFrame(synthetic_records).to_parquet(OUTPUT_PYTHON, index=False)
    pbar.close()
    print("Python Track Fully Completed and Secured!")

if __name__ == "__main__":
    synthesize_python_codenet(num_files_needed=25000)


--- Running Colab Integrated Python (CodeNet) Track ---
Downloading/Loading Project CodeNet Python split...
Starting fresh Python task.




Synthesizing Python Codes:   0%|          | 0/25000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Synthesizing Python Codes:   0%|          | 1/25000 [00:55<383:20:56, 55.20s/it]

Synthesizing Python Codes:   0%|          | 2/25000 [01:42<352:26:38, 50.76s/it]

Synthesizing Python Codes:   0%|          | 3/25000 [02:31<345:07:41, 49.70s/it]

Synthesizing Python Codes:   0%|          | 4/25000 [03:08<311:30:47, 44.87s/it]

Synthesizing Python Codes:   0%|          | 5/25000 [03:14<213:53:32, 30.81s/it]

Synthesizing Python Codes:   0%|          | 6/25000 [03:55<237:00:00, 34.14s/it]

Synthesizing Python Codes:   0%|          | 7/25000 [04:43<269:25:43, 38.81s/it]

Synthesizing Python Codes:   0%|          | 8/25000 [05:14<250:56:37, 36.15s/it]

Synthesizing Python 